# Week 24 Practice Problem 1: CatBoost Classification on Adult Income

This notebook builds a CatBoost classifier to predict whether annual income exceeds $50K using the Adult income dataset from OpenML. The workflow includes preprocessing, baseline evaluation, hyperparameter tuning, final model evaluation, and a feature-importance analysis.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

import pandas as pd
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
OUTPUT_DIR = Path.cwd() / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

X, y = fetch_openml("adult", version=2, as_frame=True, return_X_y=True)
X = X.copy()
y = y.copy()

categorical_cols = list(X.select_dtypes(include=['object', 'category']).columns)
numerical_cols = [col for col in X.columns if col not in categorical_cols]

for col in categorical_cols:
    X[col] = X[col].astype('object').fillna('Missing')
for col in numerical_cols:
    X[col] = X[col].fillna(X[col].median())

y = y.astype(str)
y = y.map({'<=50K': 0, '>50K': 1}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

cat_features = [X.columns.get_loc(col) for col in categorical_cols]
print(f'Dataset shape: {X.shape}')
print(f'Categorical features: {len(categorical_cols)}')
print(f'Numerical features: {len(numerical_cols)}')
print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')

## 1) Baseline CatBoost model

In [ ]:
baseline_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    random_seed=42,
    verbose=50,
    loss_function='Logloss',
    eval_metric='Accuracy',
)
baseline_model.fit(X_train, y_train, cat_features=cat_features)

y_pred = baseline_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f'Accuracy: {acc:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-score: {f1:.4f}')
print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))

## 2) Hyperparameter tuning with GridSearchCV

In [ ]:
estimator = CatBoostClassifier(
    iterations=10,
    random_seed=42,
    verbose=False,
    loss_function='Logloss',
    eval_metric='Accuracy',
)

param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.03, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5],
}

grid = GridSearchCV(
    estimator=estimator,
    param_grid=param_grid,
    scoring='accuracy',
    cv=3,
    n_jobs=1,
    verbose=0,
)

grid.fit(X_train, y_train, cat_features=cat_features)
print(f'Best parameters: {grid.best_params_}')
print(f'Best CV accuracy: {grid.best_score_:.4f}')

## 3) Final tuned CatBoost model

In [ ]:
final_model = CatBoostClassifier(
    iterations=500,
    random_seed=42,
    verbose=False,
    loss_function='Logloss',
    eval_metric='Accuracy',
    **grid.best_params_,
)
final_model.fit(X_train, y_train, cat_features=cat_features)

y_pred_final = final_model.predict(X_test)
final_acc = accuracy_score(y_test, y_pred_final)
final_precision = precision_score(y_test, y_pred_final, zero_division=0)
final_recall = recall_score(y_test, y_pred_final, zero_division=0)
final_f1 = f1_score(y_test, y_pred_final, zero_division=0)

print(f'Final accuracy: {final_acc:.4f}')
print(f'Final precision: {final_precision:.4f}')
print(f'Final recall: {final_recall:.4f}')
print(f'Final F1: {final_f1:.4f}')
print(confusion_matrix(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final, target_names=['<=50K', '>50K']))

In [ ]:
feature_importance = final_model.get_feature_importance(prettified=True)
top_10 = feature_importance.head(10).copy()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_10['Feature Id'][::-1], top_10['Importances'][::-1], color='#2C7FB8')
ax.set_title('Top 10 Feature Importances (CatBoost)')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'top_10_feature_importance_catboost.png', dpi=200)
plt.show()

print(top_10.to_string(index=False))

## 4) RandomForest comparison

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', SimpleImputer(strategy='median'), numerical_cols),
    ],
    remainder='drop',
)

rf_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
    ]
)
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f'RandomForest accuracy: {rf_accuracy:.4f}')

## Conclusion

This project demonstrates the strength of CatBoost on mixed-type tabular data. It handles categorical values natively, avoids one-hot encoding, and provides strong predictive performance relative to a more traditional one-hot RandomForest baseline.